# Junction Coverage Audit: How Well Does GTEx Junction Data Cover Ensembl-Annotated Splice Sites?

**Motivation.** The ablation study on M2-S v2 showed that the *junction* modality is the single
largest multimodal contributor to meta-layer PR-AUC (−0.029 when ablated, vs −0.006 for RBP).
Before investing further in junction-based features — and especially before relying on them for
M3 (novel splice-site discovery beyond annotations) — we should ask a more fundamental question:

> If GTEx junction reads (53 tissues, aggregated) can't even confirm a substantial fraction of
> splice sites that are *already annotated* in Ensembl, they certainly can't be the sole signal
> that leads the model to *novel* sites beyond annotations.

This notebook runs that audit. Three sub-questions:

1. **Annotated-site coverage.** For each Ensembl splice site on the test chromosomes, does GTEx
   see any junction supporting it? At what read depth and tissue breadth?
2. **Canonical vs alt-only breakdown.** Is coverage worse for Ensembl-alt-only sites (i.e.,
   sites in Ensembl but not MANE)? Alt sites are exactly what M2-S is trained to catch.
3. **Junctions outside annotations.** How many junction coordinates fall on positions *not*
   in any Ensembl splice site? This is the upper bound on 'novel splice sites GTEx could
   surface' — the raw material M3 would draw from.

## Data sources

| Artifact | Path | What it is |
|---|---|---|
| GTEx v8 junctions, aggregated | `data/GRCh38/junction_data/junctions_gtex_v8.parquet` | 353K unique junctions, 54 tissues collapsed |
| Ensembl splice sites | `data/ensembl/GRCh38/splice_sites_enhanced.tsv` | ~2.83M sites (incl. all transcripts) |
| MANE splice sites | `data/mane/GRCh38/splice_sites_enhanced.tsv` | Canonical-only (one transcript per gene) |

> **Note on the data layout.** Junction reads are an **experimental dataset** (GTEx RNA-seq),
> not an annotation. They live under the build-rooted, annotation-agnostic path
> `data/<build>/junction_data/` — distinct from `data/<annotation>/<build>/` which is reserved
> for annotation outputs (GTFs, splice-site TSVs, gene_features). The aggregation script
> (`scripts/data/aggregate_gtex_junctions.py`) processes raw GTEx GCT files genome-wide and
> keys the output by **Ensembl stable IDs** (e.g., `ENSG00000223972.5`); the parquet covers
> 34,080 unique genes — 97.5% in Ensembl, only ~50% with a MANE canonical transcript. So
> junction data is fundamentally Ensembl-native, and the relevant question is not *whether*
> it supports Ensembl annotation (it does, by construction) but **per-site coverage depth** —
> especially on the alt-only subset (Ensembl \\ MANE) that M2-S targets.
>
> Two namespace wrinkles to be aware of:
> - Junction `gene_id` carries a version suffix (`.5`); we strip it below.
> - MANE's `splice_sites_enhanced.tsv` uses RefSeq-style **gene symbols** (`gene-BID`)
>   rather than ENSG IDs. So any gene_id–based join with MANE would silently return
>   zero matches. We sidestep this by joining on **coordinate tuples**
>   `(chrom, position, strand, splice_type)` throughout, which is
>   annotation-namespace-agnostic.

## Coordinate conventions (the crux of this analysis)

GTEx junctions are stored as **intron coordinates** (`start`, `end` = first/last intronic
bases, 1-based). The splice sites that *bound* that intron are the **exonic** neighbors:

- On the **+ strand**: donor site = `start - 1` (last exonic base before intron), acceptor = `end + 1`.
- On the **– strand**: reversed — donor = `end + 1`, acceptor = `start - 1`.

The splice sites TSV stores a single `position` column (already the exonic coordinate),
plus `strand` and `splice_type` ∈ {`donor`, `acceptor`}. So to match a junction to a
splice site we need: *same chromosome, same strand*, *junction-derived site position equals
`splice_sites.position`*, *same splice type*.

One more wrinkle: the junction parquet uses **chr-prefixed** chromosome names (`"chr1"`)
while the Ensembl TSV uses **bare** names (`"1"`). This is the same naming mismatch that
caused the junction/RBP all-zeros bug that motivated M2-S v2 — we'll normalize here up-front.

## Tools

We use **Polars** (lazy frame API) throughout. Why polars and not pandas? Two reasons:
lazy evaluation lets us filter 2.8M rows × join 353K rows without materializing intermediate
DataFrames, and Polars is ~10x faster on this scale.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path.cwd().parents[1] if Path.cwd().name == "meta_layer" else Path.cwd()
assert (REPO / "pyproject.toml").exists(), f"expected project root, got {REPO}"

JUNCTION_PARQUET = REPO / "data/GRCh38/junction_data/junctions_gtex_v8.parquet"
ENSEMBL_SS      = REPO / "data/ensembl/GRCh38/splice_sites_enhanced.tsv"
MANE_SS         = REPO / "data/mane/GRCh38/splice_sites_enhanced.tsv"

for p in (JUNCTION_PARQUET, ENSEMBL_SS, MANE_SS):
    assert p.exists(), p

# SpliceAI test-split chromosomes (never seen during training).
# These are the ones our PR-AUCs were computed on, so coverage here
# directly explains model behavior.
TEST_CHROMS = ["1", "3", "5", "7", "9"]
TEST_CHROMS_CHR = [f"chr{c}" for c in TEST_CHROMS]

## 1. Load and peek

We read both splice-site TSVs and the junction parquet lazily. Nothing is materialized yet.

In [ ]:
junc = pl.scan_parquet(JUNCTION_PARQUET)

ensembl = pl.scan_csv(
    ENSEMBL_SS, separator="\t",
    schema_overrides={"chrom": pl.Utf8},  # 1, 2, ..., X, Y, MT
)
mane = pl.scan_csv(
    MANE_SS, separator="\t",
    schema_overrides={"chrom": pl.Utf8},
)

print("Ensembl splice sites  :", ensembl.select(pl.len()).collect().item())
print("MANE splice sites     :", mane.select(pl.len()).collect().item())
print("GTEx junctions (uniq) :", junc.select(pl.len()).collect().item())

ensembl.head(3).collect()

## 2. Normalize coordinates

Two things to fix:

1. **Chromosome naming.** Strip `chr` prefix in the junction table so it joins cleanly with Ensembl.
2. **Derive exonic splice-site positions from intronic junction coordinates.** For each junction
   we emit two rows — one for its donor side, one for its acceptor side — so we can match to the
   splice-site table directly.

**Strand logic recap** (1-based, inclusive):
- Junction = `[start, end]` is the intron.
- `+ strand`: donor = `start - 1`, acceptor = `end + 1`.
- `– strand`: donor = `end + 1`, acceptor = `start - 1`.

Problem: the GTEx junction table doesn't carry strand. We'll recover it from the Ensembl
splice-site table by joining on `gene_id` (which *is* present in the junction table). This
is a perfectly clean recovery because GTEx junction aggregation already stamped each junction
with the overlapping gene.

In practice, the cleanest match test is **symmetric**: a junction supports site `p` on strand `s`
if `p ∈ {start-1, end+1}` and the strand+type match. We'll materialize both candidate positions
and join twice.

In [ ]:
# Strip chr prefix and keep only canonical chroms (matches Ensembl naming).
junc_norm = (
    junc
    .with_columns(pl.col("chrom").str.replace("^chr", "").alias("chrom"))
    # Strip gene_id version suffix ("ENSG00000223972.5" -> "ENSG00000223972")
    .with_columns(pl.col("gene_id").str.replace(r"\..*$", "").alias("gene_id"))
)

ensembl_norm = ensembl.with_columns(pl.col("chrom").cast(pl.Utf8))

# Recover strand per junction from Ensembl splice-site table (join on gene_id).
# A gene has one canonical strand, so this is unambiguous.
gene_strand = (
    ensembl_norm.select(["gene_id", "strand"]).unique()
)
junc_strand = junc_norm.join(gene_strand, on="gene_id", how="inner")

# Emit one row per (junction, side) with the derived exonic position and the
# expected splice_type given strand.
# + strand: donor at start-1, acceptor at end+1
# – strand: donor at end+1,   acceptor at start-1
junc_sides = pl.concat([
    junc_strand.with_columns([
        pl.when(pl.col("strand") == "+").then(pl.col("start") - 1).otherwise(pl.col("end") + 1).alias("position"),
        pl.lit("donor").alias("splice_type"),
    ]),
    junc_strand.with_columns([
        pl.when(pl.col("strand") == "+").then(pl.col("end") + 1).otherwise(pl.col("start") - 1).alias("position"),
        pl.lit("acceptor").alias("splice_type"),
    ]),
]).select(["chrom", "strand", "position", "splice_type", "gene_id",
           "max_reads", "sum_reads", "n_tissues", "tissue_breadth"])

# Preview
junc_sides.head(5).collect()

## 3. Filter to the test chromosomes

We'll restrict to the SpliceAI test set — chr1, 3, 5, 7, 9 — so coverage numbers correspond
directly to the PR-AUCs we already reported. This is also much faster than scanning genome-wide.

In [ ]:
ens_test  = ensembl_norm.filter(pl.col("chrom").is_in(TEST_CHROMS))
mane_test = mane.with_columns(pl.col("chrom").cast(pl.Utf8)).filter(pl.col("chrom").is_in(TEST_CHROMS))
junc_test = junc_sides.filter(pl.col("chrom").is_in(TEST_CHROMS))

n_ens  = ens_test.select(pl.len()).collect().item()
n_mane = mane_test.select(pl.len()).collect().item()
n_junc = junc_test.select(pl.len()).collect().item()
print(f"Ensembl sites (test chroms): {n_ens:>10,}")
print(f"MANE sites    (test chroms): {n_mane:>10,}")
print(f"Junction sides (test chroms): {n_junc:>10,}  (2 per unique junction)")

## 4. Question 1: coverage of annotated sites

Left-join Ensembl sites onto junction sides. A site is **covered** if the join finds at least
one matching junction; the maximum `sum_reads` and `n_tissues` across matching junctions give
us depth and breadth.

Note: some Ensembl sites may match multiple junctions (e.g., a donor shared across several
alternative introns). We aggregate with `max()` after the join.

In [ ]:
# Unique Ensembl splice sites (a site can appear multiple times for multi-transcript rows)
ens_uniq = ens_test.select(["chrom", "position", "strand", "splice_type", "gene_id"]).unique()

coverage = (
    ens_uniq.join(
        junc_test.select(["chrom", "position", "strand", "splice_type",
                          "sum_reads", "n_tissues", "tissue_breadth"]),
        on=["chrom", "position", "strand", "splice_type"],
        how="left",
    )
    .group_by(["chrom", "position", "strand", "splice_type", "gene_id"])
    .agg([
        pl.col("sum_reads").max().alias("max_sum_reads"),
        pl.col("n_tissues").max().alias("max_n_tissues"),
        pl.col("tissue_breadth").max().alias("max_tissue_breadth"),
    ])
    .with_columns(pl.col("max_sum_reads").is_not_null().alias("covered"))
).collect()

print(f"Total unique Ensembl sites (test chroms): {len(coverage):,}")
pct_covered = coverage["covered"].mean()
print(f"Covered by any GTEx junction           : {pct_covered:.1%}")

### Breakdown by splice type and strand

In [ ]:
by_type = (
    coverage.group_by(["splice_type", "strand"])
    .agg([
        pl.len().alias("n_sites"),
        pl.col("covered").mean().alias("coverage_rate"),
        pl.col("max_sum_reads").median().alias("median_reads_when_covered"),
        pl.col("max_n_tissues").median().alias("median_tissues_when_covered"),
    ])
    .sort(["splice_type", "strand"])
)
by_type

## 5. Question 2: canonical (MANE) vs alt-only (Ensembl \ MANE)

The alt-only set is what M2-S is specifically designed to catch. If junction coverage is
already sparse here, that's a ceiling on junction-driven alt detection.

In [ ]:
mane_keys = (
    mane_test.select(["chrom", "position", "strand", "splice_type"]).unique().collect()
)
mane_keys_set = set(
    (r["chrom"], r["position"], r["strand"], r["splice_type"])
    for r in mane_keys.iter_rows(named=True)
)

coverage = coverage.with_columns(
    pl.struct(["chrom", "position", "strand", "splice_type"])
    .map_elements(lambda r: (r["chrom"], r["position"], r["strand"], r["splice_type"]) in mane_keys_set,
                  return_dtype=pl.Boolean)
    .alias("in_mane")
)

by_mane = coverage.group_by("in_mane").agg([
    pl.len().alias("n_sites"),
    pl.col("covered").mean().alias("coverage_rate"),
    pl.col("max_sum_reads").filter(pl.col("covered")).median().alias("median_reads_when_covered"),
    pl.col("max_n_tissues").filter(pl.col("covered")).median().alias("median_tissues_when_covered"),
]).sort("in_mane", descending=True)

by_mane

### Interpretation template

Read this table after running: compare the `coverage_rate` rows.

- If MANE coverage ≈ 95% and alt-only coverage ≈ 30–50%, that's a clear signal: junction
  evidence is strong for *canonical* transcripts (which dominate GTEx tissue expression) but
  rapidly thins out for alt isoforms. M2-S v2 would then owe most of its alt-site recall
  gain from junction to a *minority* of alt sites — the well-expressed ones.
- If both are ≥ 80%, junction is broadly informative and M3 has a shot at using it to extend
  *into* novel territory.
- If both are < 60%, junction data is fundamentally sparse and M3's junction channel needs
  a richer source (e.g., recount3, ENCODE long-read, per-tissue RNA-seq pools).

## 6. Depth / breadth distribution at covered sites

How much evidence, on average, does GTEx give us per covered site? A tight distribution
near 1–2 tissues suggests most support is marginal; a fat right tail says the covered sites
are robustly expressed.

In [ ]:
covered_only = coverage.filter(pl.col("covered"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(covered_only["max_n_tissues"].to_numpy(), bins=54, edgecolor="black")
axes[0].set_xlabel("# tissues supporting site (out of 54)")
axes[0].set_ylabel("# covered Ensembl sites")
axes[0].set_title("Tissue breadth at covered sites")
axes[0].grid(True, alpha=0.3)

axes[1].hist(covered_only["max_sum_reads"].clip(upper_bound=200).to_numpy(), bins=50, edgecolor="black")
axes[1].set_xlabel("sum_reads across 54 tissues (clipped at 200)")
axes[1].set_ylabel("# covered Ensembl sites")
axes[1].set_title("Read depth at covered sites")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Question 3: junction coordinates *outside* any Ensembl annotation

This is the M3 upper bound. Every junction side-position that does *not* match any Ensembl
splice site is a candidate novel splice site that GTEx could, in principle, surface.

In [ ]:
novel = (
    junc_test
    .join(
        ens_uniq.select(["chrom", "position", "strand", "splice_type"]).with_columns(pl.lit(True).alias("_in_ensembl")),
        on=["chrom", "position", "strand", "splice_type"],
        how="left",
    )
    .filter(pl.col("_in_ensembl").is_null())
    .collect()
)

total_jsides = junc_test.select(pl.len()).collect().item()
print(f"Junction sides on test chroms total : {total_jsides:,}")
print(f"Junction sides NOT in Ensembl (novel): {len(novel):,}  ({len(novel)/total_jsides:.1%})")

novel.group_by("splice_type").agg([
    pl.len().alias("n_novel_sides"),
    pl.col("sum_reads").median().alias("median_reads"),
    pl.col("n_tissues").median().alias("median_tissues"),
])

## 8. Findings (write-up after running)

Fill this section in with the printed numbers. Suggested structure:

- **Annotated-site coverage (overall):** X%
- **MANE coverage:** X%  vs  **alt-only coverage:** Y%  → gap of (X−Y) points
- **Median tissues supporting a covered site:** N (of 54)
- **Junction sides outside Ensembl (novel candidates):** Z, median depth D reads, median N tissues

### Implications for M3

M3's goal is to detect *novel* splice sites beyond annotations. The novel-junction count from
§7 is the absolute ceiling on how many GTEx can help with. If median depth and breadth there
are lower than at annotated sites, those novel candidates will be harder to distinguish from
noise — a fundamental data issue architecture can't fix.

### Possible next steps

1. **Expand junction sources** if GTEx coverage is thin — recount3, ENCODE long-read, cancer
   compendiums. Each adds tissues/cell-types not in GTEx.
2. **Per-tissue conditioning** (M2e) — a site supported by 1 tissue is signal *if* the model
   knows which tissue; an aggregated `n_tissues=1` is nearly noise without context.
3. **Soft junction evidence** — instead of `junction_has_support ∈ {0, 1}`, expose
   `tissue_breadth` as continuous input. The binary collapse throws away signal.